# 모의 데이터 개발용
기본 개발 노트북입니다. 실제 DB, Gemini, 실행 중인 HTTP 서버에 접속하지 않습니다.
임시 메뉴 6개와 리뷰 3개로 추출 → 확정 조건 검색, 리뷰 저장/재사용/수정/삭제를 확인합니다.
모의 추출은 등록된 문장만 지원하며 실제 언어 해석 품질을 평가하지 않습니다.


In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
AI_ROOT = next(p for base in (cwd, *cwd.parents) for p in (base, base / 'ai-server')
               if (p / 'app/review_summary.py').is_file())
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))


In [ ]:
from app.agent.extraction import extract_proposal
from app.llm.mock_extraction import MockExtractionModel, MESSAGES
from app.repositories.mock import MockRepository
from app.repositories.base import SearchFilters
from app.agent.direct import recommend_direct
message = MESSAGES[0]
proposal = extract_proposal(MockExtractionModel(message), message)
print('모의 추출:', proposal.model_dump())
# 아래는 자동 테스트용 확인 조건입니다. 실제 사용자 확인 화면은 별도로 검증합니다.
repo = MockRepository(AI_ROOT / 'fixtures/api_smoke_catalog.json')
result = recommend_direct(repo, SearchFilters(max_price_krw=4000, category='COFFEE', sort_by='price'))
assert [m.offer.id for m in result.recommendations] == ['m4', 'm1', 'm5']
assert result.llm_calls == 0
for m in result.recommendations:
    print(m.offer.name, m.offer.price_krw)
print('추천 PASS / 외부 API 호출 0회')


In [ ]:
from tempfile import TemporaryDirectory
from dataclasses import replace
from app.review_summary import Review, SummaryService, MockReviewSummarizer
from app.repositories.reviews import MockReviewRepository
reviews = [Review(1, 990001, '평일에는 조용해요.'),
           Review(2, 990001, '주말에는 시끄러워요.'),
           Review(3, 990001, '비공개 의견', status='HIDDEN')]
with TemporaryDirectory() as folder:
    service = SummaryService(Path(folder) / 'summary.sqlite', MockReviewSummarizer())
    review_repo = MockReviewRepository(reviews)
    summary, changed = service.refresh(990001, review_repo.list_reviews(990001))
    assert changed and summary['review_count'] == 2
    print('주제별 의견:', summary['topics'])
    print('근거 원문:', summary['opinions'])
    assert summary['topics'][0]['mixed_opinions'] is True
    assert not service.refresh(990001, review_repo.list_reviews(990001))[1]
    reviews[0] = replace(reviews[0], content='좌석이 넓어요.')
    assert service.refresh(990001, MockReviewRepository(reviews).list_reviews(990001))[1]
    reviews[0] = replace(reviews[0], deleted_at='2026-11-09')
    summary, changed = service.refresh(990001, MockReviewRepository(reviews).list_reviews(990001))
    assert changed and summary['source_review_ids'] == [2]
    print(summary)
print('리뷰 조회/저장/재사용/수정/삭제 PASS / 임시 파일 삭제 완료')


### 구조화 요약 판정
- `opinions`: 원문 발췌, 리뷰 ID, 모의 주제/긍정·부정 분류
- `topics`: 분류별 근거 ID와 서로 다른 의견 존재 여부
- `mixed_opinions`: 의견 차이가 있다는 의미이며 사실 모순을 판정하지 않습니다.
- 날짜·상황을 원문 그대로 보존합니다. 대상 리뷰 수와 실제 발췌 수를 구분합니다.
- 현재 분류는 고정 모의 문장에만 적용됩니다. 일반 문장은 other/neutral이며 LLM 평가가 아닙니다.


## 모의 리뷰 HTTP·캐시 검증
아래 테스트는 임시 SQLite와 FastAPI 테스트 클라이언트를 사용합니다.
서버 실행, 실제 DB, Gemini 없이 첫 생성/반복 캐시/빈 리뷰/인증을 검증합니다.


In [ ]:
import subprocess
checked = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                          '-p', 'test_review_summary_api.py', '-v'], cwd=AI_ROOT,
                          capture_output=True, text=True, timeout=60)
print(checked.stdout, checked.stderr)
checked.check_returncode()
